In [1]:
from typing import TypedDict, Literal
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field, ValidationError
from langgraph.graph import StateGraph, START, END
from langchain_anthropic import ChatAnthropic
load_dotenv(override=True)

/home/ubuntu/My_learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
llm = ChatAnthropic(
    api_key=os.environ.get("KEY") or os.environ.get("ANTHROPIC_API_KEY"),
    model=os.environ.get("CLAUDE_MODEL", "global.anthropic.claude-haiku-4-5-20251001-v1:0"),
    base_url=os.environ.get("ENDPOINT"),
    temperature=0.3,
    max_tokens=300,
)

In [3]:
class RouterState(TypedDict):
    question: str
    category: str
    response: str

class QuestionIn(BaseModel):
    question: str = Field(min_length=3)

class CategoryOut(BaseModel):
    category: Literal["science", "history", "general"]

class ResponseOut(BaseModel):
    response: str = Field(min_length=1)

def classify_node(state: RouterState) -> dict:
    q = QuestionIn.model_validate({"question": state.get("question", "")}).question

    prompt = (
        "Classify the question into exactly one label: science, history, or general.\n"
        "Return only one word.\n\n"
        f"Question: {q}"
    )
    raw = llm.invoke(prompt).content.strip().lower()

    if "science" in raw:
        category = "science"
    elif "history" in raw:
        category = "history"
    else:
        category = "general"

    # Validate before returning
    return CategoryOut(category=category).model_dump()

def route_by_category(
    state: RouterState,
) -> Literal["science_node", "history_node", "general_node"]:
    validated = CategoryOut.model_validate({"category": state.get("category", "general")})
    if validated.category == "science":
        return "science_node"
    if validated.category == "history":
        return "history_node"
    return "general_node"

def science_node(state: RouterState) -> dict:
    q = QuestionIn.model_validate({"question": state.get("question", "")}).question
    prompt = f"You are a science teacher. Explain clearly in 2-3 lines.\nQuestion: {q}"
    text = llm.invoke(prompt).content.strip()
    return ResponseOut(response=text).model_dump()

def history_node(state: RouterState) -> dict:
    q = QuestionIn.model_validate({"question": state.get("question", "")}).question
    prompt = f"You are a historian. Answer with brief historical context in 2-3 lines.\nQuestion: {q}"
    text = llm.invoke(prompt).content.strip()
    return ResponseOut(response=text).model_dump()

def general_node(state: RouterState) -> dict:
    q = QuestionIn.model_validate({"question": state.get("question", "")}).question
    prompt = f"You are a helpful general assistant. Answer briefly in 2-3 lines.\nQuestion: {q}"
    text = llm.invoke(prompt).content.strip()
    return ResponseOut(response=text).model_dump()

builder = StateGraph(RouterState)

builder.add_node("classify_node", classify_node)
builder.add_node("science_node", science_node)
builder.add_node("history_node", history_node)
builder.add_node("general_node", general_node)

builder.add_edge(START, "classify_node")
builder.add_conditional_edges(
    "classify_node",
    route_by_category,
    {
        "science_node": "science_node",
        "history_node": "history_node",
        "general_node": "general_node",
    },
)
builder.add_edge("science_node", END)
builder.add_edge("history_node", END)
builder.add_edge("general_node", END)

graph = builder.compile()

test_questions = [
    "Why does metal expand when heated?",
    "Who started the Renaissance in Europe?",
    "How can I manage my time better?",
    "What is photosynthesis?",
]

for q in test_questions:
    try:
        result = graph.invoke({"question": q, "category": "", "response": ""})
        print("=" * 70)
        print("Question :", q)
        print("Category :", result["category"])
        print("Answer   :", result["response"])
    except ValidationError as e:
        print(f"Validation error for question '{q}': {e}")

Question : Why does metal expand when heated?
Category : science
Answer   : # Why Metal Expands When Heated

When metal is heated, its atoms vibrate more vigorously and move faster. These energetic atoms need more space to move around, so they push farther apart from each other, causing the metal to expand overall.

This is called **thermal expansion**, and it happens because heat energy is converted into kinetic energy of the atoms.
Question : Who started the Renaissance in Europe?
Category : history
Answer   : # The Renaissance

The Renaissance didn't have a single founder, but rather emerged gradually across Italy in the 14th century, driven by scholars, artists, and wealthy patrons like the Medici family who revived classical Greek and Roman learning. Key figures like Petrarch (14th century) and later artists like Leonardo da Vinci and Michelangelo helped spread humanist ideas and artistic innovation throughout Europe during the 15th-17th centuries.
Question : How can I manage my t